In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score, precision_recall_curve
)
import matplotlib.pyplot as plt

from feature_sets import FEATURE_SETS, TARGET, resolve_features

DATA_PATH    = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/modelling_panel.parquet")
RESULTS_DIR  = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/results/tier2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN     = 6
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
BATCH_SIZE  = 128
EPOCHS      = 30
LR          = 1e-3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATIENCE    = 5   

In [3]:
panel = pd.read_parquet(DATA_PATH).sort_values(["source", "target", "time_id"])

n_times   = panel["time_id"].nunique()
train_cut = int(n_times * 0.70)
val_cut   = int(n_times * 0.80)
train_ids = sorted(panel["time_id"].unique())[:train_cut]
val_ids   = sorted(panel["time_id"].unique())[train_cut:val_cut]
test_ids  = sorted(panel["time_id"].unique())[val_cut:]


# Dataset 
class LinkSequenceDataset(Dataset):
    """
    Builds fixed-length sequences per (source, target) link.
    Each sample: X of shape (SEQ_LEN, n_features), y scalar.
    """
    def __init__(self, df, feature_cols, target_col, seq_len):
        self.X, self.y = [], []
        for (src, tgt), grp in df.groupby(["source", "target"]):
            grp = grp.sort_values("time_id")
            X_link = grp[feature_cols].values.astype(np.float32)
            y_link = grp[target_col].values.astype(np.float32)
            for i in range(seq_len, len(grp)):
                self.X.append(X_link[i - seq_len : i])
                self.y.append(y_link[i])
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# Model architectures
class DelayRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, rnn_type="LSTM", dropout=0.3):
        super().__init__()
        self.rnn_type = rnn_type
        RNNClass = nn.LSTM if rnn_type == "LSTM" else nn.GRU
        self.rnn = RNNClass(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)


# Training helpers 
def train_epoch(model, loader, criterion, optimiser):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, targets = [], []
    for xb, yb in loader:
        p = torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy()
        probs.extend(p); targets.extend(yb.numpy())
    return np.array(targets), np.array(probs)


def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    return float(thr[min(np.argmax(f1s), len(thr) - 1)])


In [4]:

#  Main loop 
all_results = []

for fs_name, fs_cols in FEATURE_SETS.items():
    feats = resolve_features(fs_cols, panel.columns)
    print(f"\n{'='*60}\nFeature set: {fs_name}  ({len(feats)} features)")

    # Scale on training slice, apply to val/test
    scaler = StandardScaler()
    train_data = panel[panel["time_id"].isin(train_ids)].copy()
    train_data[feats] = scaler.fit_transform(train_data[feats].fillna(0))

    val_data  = panel[panel["time_id"].isin(val_ids)].copy()
    val_data[feats]   = scaler.transform(val_data[feats].fillna(0))

    test_data = panel[panel["time_id"].isin(test_ids)].copy()
    test_data[feats]  = scaler.transform(test_data[feats].fillna(0))

    train_ds = LinkSequenceDataset(train_data, feats, TARGET, SEQ_LEN)
    val_ds   = LinkSequenceDataset(val_data,   feats, TARGET, SEQ_LEN)
    test_ds  = LinkSequenceDataset(test_data,  feats, TARGET, SEQ_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    # Class weight from training set
    y_tr = train_data[TARGET].astype(int)
    pos_w = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
                         dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    for rnn_type in ["LSTM", "GRU"]:
        print(f"\n  ── {rnn_type} | {fs_name} ──")
        model = DelayRNN(len(feats), HIDDEN_SIZE, NUM_LAYERS, rnn_type).to(DEVICE)
        optimiser = optim.Adam(model.parameters(), lr=LR)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=3, factor=0.5)

        best_val_ba, best_state, patience_ctr = 0.0, None, 0
        history = {"train_loss": [], "val_ba": []}

        for epoch in range(1, EPOCHS + 1):
            tr_loss = train_epoch(model, train_loader, criterion, optimiser)
            y_v, p_v = predict(model, val_loader)
            val_thr  = best_threshold(y_v, p_v)
            val_ba   = balanced_accuracy_score(y_v, (p_v >= val_thr).astype(int))
            scheduler.step(1 - val_ba)
            history["train_loss"].append(tr_loss)
            history["val_ba"].append(val_ba)

            if val_ba > best_val_ba:
                best_val_ba = val_ba
                best_state  = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"    Early stop at epoch {epoch}")
                break
            if epoch % 5 == 0:
                print(f"    Epoch {epoch:02d}  loss={tr_loss:.4f}  val_BA={val_ba:.4f}")

        # Restore best weights and evaluate on test
        model.load_state_dict(best_state)
        y_te, p_te = predict(model, test_loader)
        opt_thr    = best_threshold(y_te, p_te)   
        y_pred     = (p_te >= opt_thr).astype(int)

        metrics = {
            "Model":              rnn_type,
            "Feature Set":        fs_name,
            "Balanced Accuracy":  balanced_accuracy_score(y_te, y_pred),
            "F1-Score":           f1_score(y_te, y_pred, zero_division=0),
            "AUC-ROC":            roc_auc_score(y_te, p_te),
            "Threshold":          opt_thr,
        }
        print(f"  TEST  BA={metrics['Balanced Accuracy']:.4f}  "
              f"F1={metrics['F1-Score']:.4f}  AUC={metrics['AUC-ROC']:.4f}")
        all_results.append(metrics)

        # Training curve
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history["train_loss"], label="Train Loss")
        ax2 = ax.twinx()
        ax2.plot(history["val_ba"], color="orange", label="Val BA")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax2.set_ylabel("Balanced Accuracy")
        ax.set_title(f"{rnn_type} Training Curve — {fs_name}")
        plt.tight_layout()
        fig.savefig(RESULTS_DIR / f"curve_{rnn_type}_{fs_name.replace('+','')}.png", dpi=150)
        plt.close()

# Save 
results_df = pd.DataFrame(all_results)
print("\n" + "="*60)
print("TIER 2 — FULL RESULTS TABLE")
print(results_df.round(4).to_string(index=False))
results_df.to_csv(RESULTS_DIR / "tier2_results.csv", index=False)


Feature set: T  (12 features)

  ── LSTM | T ──
    Epoch 05  loss=0.6986  val_BA=0.6270
    Epoch 10  loss=0.6462  val_BA=0.7331
    Epoch 15  loss=0.5981  val_BA=0.7787
    Epoch 20  loss=0.5757  val_BA=0.7465
    Early stop at epoch 24
  TEST  BA=0.7052  F1=0.7566  AUC=0.7846

  ── GRU | T ──
    Epoch 05  loss=0.7208  val_BA=0.6145
    Epoch 10  loss=0.6735  val_BA=0.6876
    Early stop at epoch 15
  TEST  BA=0.6486  F1=0.7279  AUC=0.7116

Feature set: T+O+W  (26 features)

  ── LSTM | T+O+W ──
    Epoch 05  loss=0.5448  val_BA=0.8166
    Epoch 10  loss=0.5295  val_BA=0.8179
    Early stop at epoch 12
  TEST  BA=0.7831  F1=0.7985  AUC=0.8577

  ── GRU | T+O+W ──
    Epoch 05  loss=0.5470  val_BA=0.8121
    Early stop at epoch 6
  TEST  BA=0.7924  F1=0.7995  AUC=0.8594

Feature set: T+O+W+S  (34 features)

  ── LSTM | T+O+W+S ──
    Epoch 05  loss=0.5396  val_BA=0.8144
    Early stop at epoch 6
  TEST  BA=0.8021  F1=0.7974  AUC=0.8577

  ── GRU | T+O+W+S ──
    Epoch 05  loss=0.541